## Searching for signs of Clumpy Outflows in Lyman-α sources using absorption line ratios

In [ ]:
import gc

import astropy.table as aptb
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from IPython.display import display

from tangelo import constants as tgconst
from tangelo import fitting as tgfit
from tangelo import io as tgio
from tangelo import models as tgmod
from tangelo import plotting as tgplot


In [ ]:
# Load the megatable (final science-ready catalogue with absorption line measurements)
SPEC_TYPE = "weight_skysub"
tabfile = f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_{SPEC_TYPE}.fits"
with fits.open(tabfile) as tabhdul:
    megatable = aptb.Table(tabhdul[1].data)

# Quality cut: keep only sources with significant Lyα detection
megatab = megatable[(megatable['SNRR'] > 5.0) | (megatable['SNRB'] > 5.0)]

# Define absorber masks based on stacked EW significance
li_absorbers  = megatab['SNR_SiII1260']  < -5.0
hi_absorbers  = (megatab['SNR_SiIV1394']  < -5.0) & (megatab['SNR_SiIV1403']  < -5.0)
truabsorbers  = li_absorbers | hi_absorbers

print(f"Sources with any significant absorption: {np.sum(truabsorbers)} / {len(megatab)}")


In [ ]:

# Calculate the clumpiness factor for confirmed absorbers in both low- and high-ionisation lines.
#
# Low-ionisation:  EW(SiII1260) / EW(SiII1527)  — expected ~1 (optically thick) to ~5 (optically thin)
# High-ionisation: EW(SiIV1394) / EW(SiIV1403)  — expected ~1 (optically thick) to ~2 (optically thin)
ewratios_lo, ewratios_hi = {}, {}
hilines, lolines = ['SiIV1394', 'SiIV1403'], ['SiII1260', 'SiII1527']
lr_lo = tgconst.wavedict['SiII1527'] / tgconst.wavedict['SiII1260']
lr_hi = tgconst.wavedict['SiIV1403'] / tgconst.wavedict['SiIV1394']

N_SAMP = 1000           # samples drawn from parameter posteriors for EW ratio distribution
MF_SIGMA_THRESH = 3.0   # minimum MF sigma to accept a detection
SII1527_MF_THRESH = 5.0 # stricter threshold for SiII1527, whose weakness makes ratios poorly constrained
FIT_RANGE = 50          # continuum_buffer passed to fit_line (Å either side of line centre)

# Bootstrap parameters passed to fit_line for error estimation
bootstrap_params = {
    'niter': 1000,
    'autocorrelation': True,
    'errfunc': 'stddev_7',
    'baseline_order': 1,
}

# Per-source observed-wavelength exclusion zones to mask from fitting and matched-filter.
# Keys: "{CLUSTER}.{iden}", values: list of (lo_wavelength, hi_wavelength) tuples.
# Example: 'MACS1206.SP3432': [(5628, 5636), (5637, 5645)]
exclusion_zones = {
    'MACS1206.SP2684': [(5628, 5636), (5637, 5645)],
    'MACS1206.SP2761': [(5628, 5636), (5637, 5645), (5043, 5051), (5056,5064), (5064,5072), (5098, 5106)],
    'MACS1206.P8042': [(5628, 5636), (5637, 5645)],
    'SMACS2031.SP74': [(6832, 6840), (6843, 6851)],
}

for row in megatab[truabsorbers][:-2]:
    src_name = f"{row['CLUSTER']} {row['iden']}"
    print(f"\n── {src_name} ──")
    targtab = tgio.load_spec(row['CLUSTER'], row['iden'], row['idfrom'])
    if targtab is None:
        print("  No spectrum found, skipping.")
        continue
    wavax    = targtab['wave']
    targspec = targtab['spec']
    targerr  = targtab['spec_err']

    # Apply per-source exclusion zones: set errors to inf so generate_spec_mask excludes
    # those channels from the fit, and fit_line's gap-detection adds them to the MF exclusion.
    exc_key = f"{row['CLUSTER']}.{row['iden']}"
    targerr_masked = targerr.copy()
    for lo, hi in exclusion_zones.get(exc_key, []):
        targerr_masked[(wavax >= lo) & (wavax <= hi)] = np.inf

    # --- Low-ionisation: SiII1260 (MF-gated) + SiII1527 (tied to SiII1260 redshift) ---
    if row[f'SNR_{lolines[0]}'] < -3.0:
        obsw_1260 = row[f'LPEAK_{lolines[0]}']
        fig_lo, axs_lo = plt.subplots(2, 1, figsize=(4, 5), facecolor='w')

        res_1260 = tgfit.fit_line(
            wavax, targspec, targerr_masked, 'SiII1260',
            {'LPEAK': obsw_1260, 'FLUX': row[f'FLUX_{lolines[0]}'],
             'FWHM': row[f'FWHM_{lolines[0]}'], 'CONT': row[f'CONT_{lolines[0]}'],
             'SLOPE': row[f'SLOPE_{lolines[0]}']},
            bounds={'LPEAK': (obsw_1260 - 10., obsw_1260 + 10.), 'FLUX': (-np.inf, 0.), 'FWHM': (2.0, 20)},
            continuum_buffer=FIT_RANGE, plot_result=True, ax_in=axs_lo[0],
            bootstrap_params=bootstrap_params,
        )

        if not res_1260:
            print("  [LO] SiII1260 fit failed.")
            display(fig_lo); plt.close(fig_lo)
        else:
            mf_ok_1260 = res_1260['mf_sigma'][0] >= MF_SIGMA_THRESH
            print(f"  [LO] MF {lolines[0]}: {res_1260['mf_result'][0]['message']} "
                  f"{'✓' if mf_ok_1260 else '✗'}")

            # Fit SiII1527 with centre locked to the same redshift as the fitted SiII1260 centroid.
            # Bounds ±0.5 Å keep all MC samples at the same redshift; no independent MF check.
            obsw_1527 = res_1260['param_dict']['LPEAK'] * lr_lo
            res_1527 = tgfit.fit_line(
                wavax, targspec, targerr_masked, 'SiII1527',
                {'LPEAK': obsw_1527, 'FLUX': row[f'FLUX_{lolines[0]}'],
                 'FWHM': row[f'FWHM_{lolines[0]}'], 'CONT': row[f'CONT_{lolines[0]}'],
                 'SLOPE': row[f'SLOPE_{lolines[0]}']},
                bounds={'LPEAK': (obsw_1527 - 0.5, obsw_1527 + 0.5), 'FLUX': (-np.inf, 0.), 'FWHM': (2.0, 20)},
                continuum_buffer=FIT_RANGE, plot_result=True, ax_in=axs_lo[1],
                bootstrap_params=bootstrap_params,
            )
            display(fig_lo); plt.close(fig_lo)

            if mf_ok_1260:
                if not res_1527:
                    print("  [LO] SiII1527 fit failed; EW ratio not stored.")
                else:
                    mf_ok_1527 = res_1527['mf_sigma'][0] >= SII1527_MF_THRESH
                    print(f"  [LO] MF {lolines[1]}: {res_1527['mf_result'][0]['message']} "
                          f"{'✓' if mf_ok_1527 else '✗'} (threshold {SII1527_MF_THRESH:.0f}σ)")
                    if not mf_ok_1527:
                        print(f"  [LO] SiII1527 below {SII1527_MF_THRESH:.0f}σ; EW ratio not stored.")
                    else:
                        p1, e1 = res_1260['param_dict'], res_1260['error_dict']
                        p2, e2 = res_1527['param_dict'], res_1527['error_dict']
                        flux1 = np.random.normal(p1['FLUX'], e1['FLUX'], N_SAMP)
                        cont1 = np.random.normal(p1['CONT'], e1['CONT'], N_SAMP)
                        flux2 = np.random.normal(p2['FLUX'], e2['FLUX'], N_SAMP)
                        cont2 = np.random.normal(p2['CONT'], e2['CONT'], N_SAMP)
                        ew_ratio = (flux1 / cont1) / (flux2 / cont2)
                        print(f"  [LO] EW ratio: median={np.nanmedian(ew_ratio):.2f}  "
                              f"[{np.percentile(ew_ratio, 0.2):.2f}, {np.percentile(ew_ratio, 99.8):.2f}]")
                        ewratios_lo[src_name] = ew_ratio
        gc.collect()

    # --- High-ionisation: SiIV1394 + SiIV1403 (fitted as a doublet by fit_line) ---
    if row[f'SNR_{hilines[0]}'] < -3.0:
        obsw_1394 = row[f'LPEAK_{hilines[0]}']
        fig_hi, ax_hi = plt.subplots(1, 1, figsize=(4, 3), facecolor='w')

        res_hi = tgfit.fit_line(
            wavax, targspec, targerr_masked, 'SiIV1394',
            {'LPEAK': obsw_1394, 'FLUX': row[f'FLUX_{hilines[0]}'],
             'FWHM': row[f'FWHM_{hilines[0]}'], 'FLUX2': row[f'FLUX_{hilines[1]}'],
             'CONT': row[f'CONT_{hilines[0]}'], 'SLOPE': row[f'SLOPE_{hilines[0]}']},
            bounds={'LPEAK': (obsw_1394 - 10., obsw_1394 + 10.),
                    'FLUX': (-np.inf, 0.), 'FLUX2': (-np.inf, 0.),
                    'FWHM': (2.0, 20)},
            continuum_buffer=FIT_RANGE, plot_result=True, ax_in=ax_hi,
            bootstrap_params=bootstrap_params,
        )
        display(fig_hi); plt.close(fig_hi)

        if not res_hi:
            print("  [HI] SiIV1394/1403 doublet fit failed.")
        else:
            mf_ok_hi = [res_hi['mf_sigma'][0] >= MF_SIGMA_THRESH,
                        res_hi['mf_sigma'][1] >= MF_SIGMA_THRESH]
            print(f"  [HI] MF {hilines[0]}: {res_hi['mf_result'][0]['message']} "
                  f"{'✓' if mf_ok_hi[0] else '✗'}")
            print(f"  [HI] MF {hilines[1]}: {res_hi['mf_result'][1]['message']} "
                  f"{'✓' if mf_ok_hi[1] else '✗'}")

            if all(mf_ok_hi):
                p, e = res_hi['param_dict'], res_hi['error_dict']
                # Sample from posteriors; continuum is evaluated at doublet midpoint
                flux1 = np.random.normal(p['FLUX'],  e['FLUX'],  N_SAMP)
                flux2 = np.random.normal(p['FLUX2'], e['FLUX2'], N_SAMP)
                cont  = np.random.normal(p['CONT'],  e['CONT'],  N_SAMP)
                slope = np.random.normal(p['SLOPE'], e['SLOPE'], N_SAMP)
                lpeak = np.random.normal(p['LPEAK'], e['LPEAK'], N_SAMP)
                mid   = lpeak * (1 + lr_hi) / 2
                cont_at_1394 = cont + slope * (lpeak          - mid)
                cont_at_1403 = cont + slope * (lpeak * lr_hi  - mid)
                ew_ratio = (flux1 / cont_at_1394) / (flux2 / cont_at_1403)
                print(f"  [HI] EW ratio: median={np.nanmedian(ew_ratio):.2f}  "
                      f"[{np.percentile(ew_ratio, 0.15):.2f}, {np.percentile(ew_ratio, 99.85):.2f}]")
                ewratios_hi[src_name] = ew_ratio
            else:
                failed = [hilines[k] for k in range(2) if not mf_ok_hi[k]]
                print(f"  [HI] MF failed for {', '.join(failed)}; EW ratio not stored.")
        gc.collect()


In [ ]:
# Violin plots of the EW ratios to establish clumpiness
fig, ax = plt.subplots(1, 2, figsize=(6, 4), facecolor='w', sharey=True)

# List of sources to exclude
exclude_sources = [
    # 'MACS1206 SP3432', 
    # 'MACS0416NE SP1580'
]

# Collect all unique source names that have results in either dict, excluding problematic sources
all_source_names = sorted(
    (set(ewratios_lo.keys()) | set(ewratios_hi.keys())) - set(exclude_sources)
)
y_positions = np.arange(len(all_source_names))
obj_to_pos  = {name: pos for pos, name in enumerate(all_source_names)}

# Accumulate data arrays for violin plots
ewratios_lo_clean, ewratios_hi_clean = [], []
positions_lo, positions_hi = [], []

for src_name, pos in obj_to_pos.items():
    if src_name in ewratios_lo and len(ewratios_lo[src_name]) > 4:
        ewratios_lo_clean.append(ewratios_lo[src_name])
        positions_lo.append(pos)
    if src_name in ewratios_hi and len(ewratios_hi[src_name]) > 4:
        ewratios_hi_clean.append(ewratios_hi[src_name])
        positions_hi.append(pos)

ewratios_lo_arr = np.array(ewratios_lo_clean)
ewratios_hi_arr = np.array(ewratios_hi_clean)
positions_lo    = np.array(positions_lo)
positions_hi    = np.array(positions_hi)

# Create violin plots
violin_lo = ax[0].violinplot(ewratios_lo_arr.T, positions=positions_lo,
                             vert=False, showextrema=False)
violin_hi = ax[1].violinplot(ewratios_hi_arr.T, positions=positions_hi,
                             vert=False, showextrema=False)

# Style the violins
for pc in violin_lo['bodies']:
    pc.set_facecolor('black')
    pc.set_alpha(0.6)
for pc in violin_hi['bodies']:
    pc.set_facecolor('black')
    pc.set_alpha(0.6)

# Reference lines
ax[0].axvline(1., linestyle=':', color='blue', label='optically thick')
ax[0].axvline(5., linestyle=':', color='red',  label='optically thin')
ax[1].axvline(1., linestyle=':', color='blue')
ax[1].axvline(2., linestyle=':', color='red')

# Ticks and labels
ax[0].set_yticks(y_positions)
ax[0].set_yticklabels(all_source_names)
ax[0].set_xticks([0, 1, 2, 3, 4, 5, 6])
ax[1].set_xticks([0, 1, 2, 3, 4, 5, 6])
ax[0].set_xlabel('EW(SiII1260)/EW(SiII1527)')
ax[1].set_xlabel('EW(SiIV1394)/EW(SiIV1403)')
ax[0].set_xlim(0., 5.5)
ax[1].set_xlim(0., 5.5)

plt.setp(ax[0].get_yticklabels(), rotation=0, fontsize=10)
ax[0].legend(loc='lower right')

plt.tight_layout()
fig.savefig("./plots/absorber_clumpiness.pdf", bbox_inches='tight')
plt.show()
plt.close()
